# Create the shared Wealth ML setup secret

Run this once per workload account and region using an administrator/operator
identity authorized to create the secret. It reads the working
`notebooks/config.json`; keep that local file out of source control.
It stores platform configuration, not AWS credentials.

The shared fields are bucket, S3 prefix, execution-role ARN, image URI,
primary key, and either Registry group prefix or the exact shared group.
Optional tracking/completion settings are included if configured.
Each scientist keeps their own namespace, data paths and metadata file.

Existing identical settings are reused. Different settings are reported;
this notebook does not overwrite an existing secret. Deliberate later
changes can be made in Secrets Manager.


## Choose the account environment

- Use the current runtime identity and region; no credentials are copied.
- Find the application checkout from the current notebook directory.
- Load the setup helper and show its location before any AWS call.


In [ ]:
from pathlib import Path
import json
import sys
import boto3

SECRET_NAME = "wsf-mlops-setup"
REGION = None  # Use this environment's AWS region, or set the workload region explicitly.

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src").is_dir())
SOURCE = ROOT / "notebooks/source/setup"
sys.path.insert(0, str(SOURCE))
import platform_settings
assert Path(platform_settings.__file__).resolve() == (SOURCE / "platform_settings.py").resolve(), (
    "Restart the kernel: setup helpers were imported from another checkout."
)
session = boto3.Session(region_name=REGION)
print("Setup source:", SOURCE)
print("Secret name:", SECRET_NAME, "| Region:", session.region_name or "not configured")

KMS_KEY_ID = None  # Set the approved customer-managed key ARN if required.


## Create or verify the shared configuration

- Read only the approved shared fields from local config.json.
- Verify the execution role belongs to the caller's account.
- Create the secret if absent, then read back the exact created version.
- Return its ARN/version without printing the stored configuration.

Requires Secrets Manager GetSecretValue/CreateSecret. A customer-managed
KMS key also requires its encryption/decryption permissions.


In [ ]:
result = platform_settings.create_setup_secret(
    ROOT, session, secret_name=SECRET_NAME, kms_key_id=KMS_KEY_ID,
)
print(json.dumps(result, indent=2))
print("Ready: new checkouts can run notebooks/00_load_setup.ipynb.")
